# Developing an API for Excel

In this practice you will analyse which customer segments responded to a skin clinic's marketing campaign, publish the results through a FastAPI endpoint designed for Excel, and connect Excel to the endpoint with Power Query, so that business users can refresh the tables with one button.

Everything here uses only what Lesson 3 covers: building an Excel-friendly API with FastAPI, deploying it, importing web data with Power Query, and refreshing it with a VBA macro.

## Background

A skin clinic ran a marketing campaign targeting 10,000 customers. The clinic wants to know which customer segments are more likely to respond, based on demographics and purchase behaviour. The insights will help it target future campaigns and increase campaign effectiveness.

**Before you start:** install the libraries with `pip install fastapi uvicorn httpx pandas lxml`. `httpx` and `lxml` are needed only for testing inside this notebook. Steps 6 and 7 need Microsoft Excel for Windows or Mac with the **Developer** tab enabled.

## Data Description

This practice uses **skin clinic campaign.csv** (10,000 customers, one row per customer), included in the download.

| Column Name | Description |
|---|---|
| CustID | Unique customer identifier |
| Gender | Female or Male |
| AgeGroup | <30, 30-50 or >50 |
| Purchase_Last_Quarter | Whether the customer made a purchase in the last quarter (Yes/No) |
| Unique_Products_Purchased | Number of different products bought in the last year (1 to 15) |
| Response_to_Campaign | **Target variable:** whether the customer responded to the campaign (Yes/No) |

In [1]:
import pandas as pd

df = pd.read_csv("skin clinic campaign.csv")
print(df.shape)
df.head()

(10000, 6)


,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes


## 1. Load and Prepare the Data

We convert **Response_to_Campaign** to 1 (Yes) and 0 (No). The mean of this column is then the response rate. We also group **Unique_Products_Purchased** into three bands: 1-4, 5-8 and more than 8.

In [2]:
df["Response"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

def categorize_products(x):
    if x <= 4:
        return "1-4"
    elif x <= 8:
        return "5-8"
    else:
        return ">8"

df["Product_Category"] = df["Unique_Products_Purchased"].apply(categorize_products)

print("Overall response rate:", round(df["Response"].mean() * 100, 1), "%")

Overall response rate: 39.9 %


## 2. Response Rate by Segment

A helper function returns the response rate (%) for each level of a column. We use it for the four tables the clinic asked for: gender, age group, purchase in the last quarter, and product usage.

In [3]:
def response_table(group_col):
    result = df.groupby(group_col)["Response"].mean().reset_index()
    result["Response Rate (%)"] = (result["Response"] * 100).round(2)
    return result.drop(columns=["Response"])

for col in ["Gender", "AgeGroup", "Purchase_Last_Quarter", "Product_Category"]:
    display(response_table(col))

,Gender,Response Rate (%)
0,Female,43.77
1,Male,34.08


,AgeGroup,Response Rate (%)
0,30-50,47.06
1,<30,32.93
2,>50,32.60


,Purchase_Last_Quarter,Response Rate (%)
0,No,21.69
1,Yes,49.65


,Product_Category,Response Rate (%)
0,1-4,18.08
1,5-8,38.48
2,>8,52.00


**Interpretation:** the overall response rate is about 40%. Customers aged 30-50, those who bought in the last quarter, and those who buy more than 8 different products respond best (around 47-52%). Customers who bought 1-4 products respond least (about 18%). Future campaigns should prioritise recent, multi-product customers in the 30-50 age group.

## 3. Build an Excel-Friendly API

Excel's Power Query web connector looks for HTML tables in a page. To make each table easy to detect, the endpoint returns **only the four tables**, with no headings or styling.

In [4]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

app = FastAPI()

@app.get("/campaign-analysis", response_class=HTMLResponse)
def campaign_analysis():
    gender = response_table("Gender").to_html(index=False)
    age = response_table("AgeGroup").to_html(index=False)
    purchase = response_table("Purchase_Last_Quarter").to_html(index=False)
    product = response_table("Product_Category").to_html(index=False)

    return f"""
    {gender}
    {age}
    {purchase}
    {product}
    """

## 4. Test the Endpoint the Way Excel Reads It

We call the endpoint with FastAPI's `TestClient`, then use `pd.read_html()` to find every table in the response. This mirrors what Power Query does: if pandas finds four tables, Excel's Navigator will list four tables, **Table 0** to **Table 3**.

In [5]:
from io import StringIO
from fastapi.testclient import TestClient

client = TestClient(app)
response = client.get("/campaign-analysis")
print("Status code:", response.status_code)

tables = pd.read_html(StringIO(response.text))
print("Tables found:", len(tables))

for i, table in enumerate(tables):
    print(f"\nTable {i}")
    print(table.to_string(index=False))

Status code: 200
Tables found: 4

Table 0
Gender  Response Rate (%)
Female              43.77
  Male              34.08

Table 1
AgeGroup  Response Rate (%)
   30-50              47.06
     <30              32.93
     >50              32.60

Table 2
Purchase_Last_Quarter  Response Rate (%)
                   No              21.69
                  Yes              49.65

Table 3
Product_Category  Response Rate (%)
             1-4              18.08
             5-8              38.48
              >8              52.00


## 5. Save the App and Deploy on Render

These cells create a project folder, **campaign-api**, copy the dataset into it, and write **main.py** and **requirements.txt** there, so the lesson's own example files (including its main.py) are not overwritten. Then deploy as in Lesson 2:

1. Put the contents of the **campaign-api** folder (**main.py**, **requirements.txt** and **skin clinic campaign.csv**) in a GitHub repository.
2. On Render, create a new **Web Service** from that repository.
3. Build command: `pip install -r requirements.txt`
4. Start command: `uvicorn main:app --host 0.0.0.0 --port $PORT`
5. Open `https://<your-app-name>.onrender.com/campaign-analysis` to check that the endpoint is publicly accessible.

In [6]:
from pathlib import Path
import shutil

project = Path("campaign-api")
project.mkdir(exist_ok=True)
shutil.copy("skin clinic campaign.csv", project)
print("Project folder ready:", sorted(f.name for f in project.iterdir()))

Project folder ready: ['skin clinic campaign.csv']


In [7]:
%%writefile campaign-api/main.py
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import pandas as pd

app = FastAPI()

# Load the data
df = pd.read_csv("skin clinic campaign.csv")

# Convert the response to 1 (Yes) and 0 (No)
df["Response"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

# Group the number of unique products purchased
def categorize_products(x):
    if x <= 4:
        return "1-4"
    elif x <= 8:
        return "5-8"
    else:
        return ">8"

df["Product_Category"] = df["Unique_Products_Purchased"].apply(categorize_products)

# Response rate (%) for each level of a column
def response_table(group_col):
    result = df.groupby(group_col)["Response"].mean().reset_index()
    result["Response Rate (%)"] = (result["Response"] * 100).round(2)
    return result.drop(columns=["Response"])


# Excel-friendly endpoint: only the four HTML tables, no headings or styling,
# so that Power Query detects each one as a separate table
@app.get("/campaign-analysis", response_class=HTMLResponse)
def campaign_analysis():
    gender = response_table("Gender").to_html(index=False)
    age = response_table("AgeGroup").to_html(index=False)
    purchase = response_table("Purchase_Last_Quarter").to_html(index=False)
    product = response_table("Product_Category").to_html(index=False)

    return f"""
    {gender}
    {age}
    {purchase}
    {product}
    """

Writing campaign-api/main.py


In [8]:
%%writefile campaign-api/requirements.txt
fastapi
uvicorn
pandas

Writing campaign-api/requirements.txt


## 6. Connect Excel with Power Query

1. In Excel, choose **Data → Get Data → From Other Sources → From Web**.
2. Paste `https://<your-app-name>.onrender.com/campaign-analysis` and click **OK**.
3. In the **Navigator**, tick **Select multiple items**, then tick **Table 0** to **Table 3** (gender, age group, last-quarter purchase and product usage).
4. Choose **Load To...** and place the tables on a worksheet.
5. Optional: in **Query Properties**, tick **Refresh data when opening the file**.

## 7. Add a Refresh Button

1. Open the VBA editor with **Alt + F11** (Windows) or **Tools → Macro → Visual Basic Editor** (Mac), and choose **Insert → Module**.
2. Paste the macro below.
3. Back in the worksheet, choose **Developer → Insert → Form Controls → Button**, draw the button, and assign it the macro **RefreshCampaignAnalysis**.
4. Save the workbook as a macro-enabled file (**.xlsm**).

```vba
Sub RefreshCampaignAnalysis()
    ' Re-runs every Power Query connection in the workbook
    ThisWorkbook.RefreshAll
    MsgBox "Campaign analysis updated!"
End Sub
```

Clicking the button re-runs every Power Query connection, so the tables always show the latest results from the API.

## Summary

- An Excel-friendly API returns only HTML tables, with no headings or styling, so Power Query detects each table.
- `pd.read_html()` checks the endpoint the same way Excel reads it: four tables found means four tables in the Navigator.
- Power Query's **From Web** connector loads the tables from the deployed endpoint into a worksheet.
- A VBA macro calling `ThisWorkbook.RefreshAll`, assigned to a button, refreshes the results with one click.
- Business users get the latest analysis in Excel without running any Python.